# PRACTICE 3 – GET STARTED WITH HUGGING FACE
**Môn học:** Deep Learning  
**Thực hiện:** Bài tập Practice 3 – Hugging Face Transformers & Fine-tuning  
**Môi trường:** Jupyter Notebook (`.ipynb`)  

---

## 1. Chuẩn bị môi trường (Environment Setup)
Mục đích: Kiểm tra phiên bản Python, PyTorch và cài đặt/kiểm tra các thư viện cần thiết từ Hugging Face ecosystem (`transformers`, `datasets`, `evaluate`, `accelerate`, `scikit-learn`).

In [1]:
# Cell 1 - Kiểm tra môi trường & Cài đặt thư viện
%pip install -q transformers datasets evaluate accelerate torch scikit-learn

import sys
import torch
import transformers
import datasets
import evaluate

print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Evaluate version: {evaluate.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("Sử dụng CPU để huấn luyện.")

C:\Users\Admin\Documents\GitHub\Practice 3\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


Python version: 3.10.20
PyTorch version: 2.13.0+cpu
Transformers version: 5.15.1
Datasets version: 5.0.1
Evaluate version: 0.4.6
CUDA Available: False
Sử dụng CPU để huấn luyện.


---
## 2. EXERCISE 1 – Sentiment Analysis với Hugging Face Pre-trained Model

### Mục tiêu Exercise 1:
Thực hiện phân tích cảm xúc (Sentiment Analysis) một câu tiếng Anh bất kỳ sử dụng model đã được huấn luyện sẵn (`distilbert-base-uncased-finetuned-sst-2-english`) trên Hugging Face Hub thông qua hàm `pipeline`.

Luồng xử lý (Pipeline Architecture):
```text
Input sentence ──> Tokenizer ──> Token IDs ──> Pre-trained Model ──> Prediction ──> POSITIVE / NEGATIVE
```

### Bước 1 & 2: Import thư viện & Khởi tạo Sentiment Analysis Pipeline

`pipeline` trong thư viện `transformers` là một abstraction layer giúp đơn giản hóa việc load model, tokenizer và tiền xử lý/hậu xử lý dữ liệu chỉ với vài dòng code.

In [2]:
# Cell 2 - Import pipeline & Load pre-trained sentiment analysis model
from transformers import pipeline

print("Đang khởi tạo sentiment-analysis pipeline...")
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
print("Load model thành công!")

Đang khởi tạo sentiment-analysis pipeline...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Load model thành công!


### Bước 3 & 4: Phân tích cảm xúc với 5 câu thử nghiệm

Ta sẽ kiểm tra mô hình với 5 câu mang sắc thái khác nhau:
1. Câu rất tích cực
2. Câu rất tiêu cực
3. Câu trung tính / mơ hồ
4. Câu chứa từ phủ định (Negative word)
5. Câu tự chọn phức tạp

In [3]:
# Cell 3 - Chạy Sentiment Analysis trên 5 câu đại diện
test_sentences = [
    "I absolutely love learning Deep Learning with Hugging Face, it is fantastic!", # 1. Rất tích cực
    "The service was terrible, the room was dirty, and I hated my stay.",           # 2. Rất tiêu cực
    "The weather today is cloudy with a chance of rain later in the afternoon.",     # 3. Trung tính
    "The movie was not bad at all, in fact I quite enjoyed it.",                   # 4. Phủ định
    "Although the code took a long time to train, the final accuracy was amazing!"  # 5. Tự chọn
]

print("=== KẾT QUẢ PHÂN TÍCH CẢM XÚC (SENTIMENT ANALYSIS) ===")
for i, sentence in enumerate(test_sentences, 1):
    result = classifier(sentence)[0]
    label = result['label']
    score = result['score']
    print(f"\nCâu {i}: '{sentence}'")
    print(f"   => Nhãn dự đoán (Label): {label}")
    print(f"   => Độ tin cậy (Score):   {score:.4f} ({score*100:.2f}%)")

=== KẾT QUẢ PHÂN TÍCH CẢM XÚC (SENTIMENT ANALYSIS) ===

Câu 1: 'I absolutely love learning Deep Learning with Hugging Face, it is fantastic!'
   => Nhãn dự đoán (Label): POSITIVE
   => Độ tin cậy (Score):   0.9999 (99.99%)

Câu 2: 'The service was terrible, the room was dirty, and I hated my stay.'
   => Nhãn dự đoán (Label): NEGATIVE
   => Độ tin cậy (Score):   0.9997 (99.97%)

Câu 3: 'The weather today is cloudy with a chance of rain later in the afternoon.'
   => Nhãn dự đoán (Label): NEGATIVE
   => Độ tin cậy (Score):   0.9347 (93.47%)

Câu 4: 'The movie was not bad at all, in fact I quite enjoyed it.'
   => Nhãn dự đoán (Label): POSITIVE
   => Độ tin cậy (Score):   0.9998 (99.98%)

Câu 5: 'Although the code took a long time to train, the final accuracy was amazing!'
   => Nhãn dự đoán (Label): POSITIVE
   => Độ tin cậy (Score):   0.9999 (99.99%)


### Bước 5: Tìm hiểu cơ chế Tokenization (Token & Token ID)

Để mô hình NLP có thể hiểu câu văn, văn bản cần trải qua quá trình **Tokenization**:
1. **Text**: Chuỗi ký tự thô.
2. **Tokens**: Các từ/từ phụ (subwords) được tách ra.
3. **Token IDs**: Các số nguyên đại diện cho từng token trong từ điển (Vocabulary) của mô hình.
4. **`input_ids`**: Mã số đại diện cho chuỗi token bao gồm cả token đặc biệt (như `[CLS]`, `[SEP]`).
5. **`attention_mask`**: Mảng nhị phân (`1` cho token thật, `0` cho padding) thông báo cho mô hình biết vị trí nào cần quan tâm.

In [4]:
# Cell 4 - Phân tích Tokenization chi tiết với AutoTokenizer
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)

sample_sentence = "I love learning Deep Learning with Hugging Face!"

# 1. Tách chuỗi thành danh sách các Token (Subwords)
tokens = tokenizer.tokenize(sample_sentence)

# 2. Chuyển đổi Tokens thành Token IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)

# 3. Mã hóa đầy đủ (bao gồm special tokens như [CLS], [SEP], attention_mask)
encoded_input = tokenizer(sample_sentence)

print("=== CHI TIẾT TỪNG BƯỚC TOKENIZATION ===")
print(f"Câu gốc (Text):             '{sample_sentence}'")
print(f"Danh sách Token:            {tokens}")
print(f"Danh sách Token ID:         {token_ids}")
print(f"Output của tokenizer():     {list(encoded_input.keys())}")
print(f"Input IDs (có CLS & SEP):   {encoded_input['input_ids']}")
print(f"Attention Mask:             {encoded_input['attention_mask']}")

# Giải mã lại Token IDs về chuỗi gốc
decoded_text = tokenizer.decode(encoded_input['input_ids'])
print(f"Giải mã ngược (Decode):     '{decoded_text}'")

=== CHI TIẾT TỪNG BƯỚC TOKENIZATION ===
Câu gốc (Text):             'I love learning Deep Learning with Hugging Face!'
Danh sách Token:            ['i', 'love', 'learning', 'deep', 'learning', 'with', 'hugging', 'face', '!']
Danh sách Token ID:         [1045, 2293, 4083, 2784, 4083, 2007, 17662, 2227, 999]
Output của tokenizer():     ['input_ids', 'token_type_ids', 'attention_mask']
Input IDs (có CLS & SEP):   [101, 1045, 2293, 4083, 2784, 4083, 2007, 17662, 2227, 999, 102]
Attention Mask:             [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Giải mã ngược (Decode):     '[CLS] i love learning deep learning with hugging face! [SEP]'


---
## 3. EXERCISE 2 – Fine-tuning Pre-trained Model cho Binary Text Classification

### Mục tiêu Exercise 2:
Lấy mô hình nền chưa fine-tune cảm xúc (`distilbert-base-uncased`) và huấn luyện lại (fine-tune) trên tập dữ liệu IMDb Movie Reviews để phân loại 2 lớp sentiment (0: Negative, 1: Positive).

Luồng xử lý (Fine-tuning Workflow):
```text
Dataset (IMDb) ──> Tokenize & Pad ──> Base Model + Head ──> Trainer.train() ──> Evaluate Metrics
```

### Bước 1 & 2: Load IMDb Dataset & Rút gọn tập dữ liệu
Tập dữ liệu IMDb gồm 50,000 đánh giá phim. Vì thực hành trên môi trường CPU, ta rút gọn tập train (200 mẫu) và test (50 mẫu) để đảm bảo thời gian chạy tối ưu (vài chục giây) mà vẫn hoàn thành đầy đủ quy trình fine-tuning và đánh giá mô hình.

In [5]:
# Cell 5 - Load & Subset IMDb Dataset
from datasets import load_dataset

print("Đang tải tập dữ liệu IMDb (stanfordnlp/imdb)...")
dataset = load_dataset("stanfordnlp/imdb")

# Lấy subset nhỏ để thực hành nhanh trên CPU (200 train, 50 test)
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(200))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(50))

print(f"Kích thước tập Train rút gọn: {len(small_train_dataset)} mẫu")
print(f"Kích thước tập Test rút gọn:  {len(small_test_dataset)} mẫu")

# In thử 1 sample đầu tiên
sample = small_train_dataset[0]
print("\n--- MẪU DỮ LIỆU ĐẦU TIÊN (SAMPLE 0) ---")
print(f"Text (200 ký tự đầu): {sample['text'][:200]}...")
print(f"Label (0=Negative, 1=Positive): {sample['label']}")

Đang tải tập dữ liệu IMDb (stanfordnlp/imdb)...


Kích thước tập Train rút gọn: 200 mẫu
Kích thước tập Test rút gọn:  50 mẫu

--- MẪU DỮ LIỆU ĐẦU TIÊN (SAMPLE 0) ---
Text (200 ký tự đầu): There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F...
Label (0=Negative, 1=Positive): 1


### Bước 3 & 4: Load Tokenizer & Tiền xử lý dữ liệu (Preprocessing)
Sử dụng `AutoTokenizer` của `distilbert-base-uncased`. Viết hàm `tokenize_function` để chuyển đổi câu văn bản thành tensor với `padding="max_length"`, `truncation=True` và `max_length=128`.

In [6]:
# Cell 6 - Tokenize toàn bộ Dataset với map()
from transformers import AutoTokenizer

base_model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("Đang tiền xử lý (tokenize) tập Train và Test...")
tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_test = small_test_dataset.map(tokenize_function, batched=True)
print("Hoàn tất tiền xử lý dữ liệu!")

Đang tiền xử lý (tokenize) tập Train và Test...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Hoàn tất tiền xử lý dữ liệu!


### Bước 5 & 6: Load Pre-trained Model & Khởi tạo Training Arguments
Ta load `AutoModelForSequenceClassification` với `num_labels=2`. Khi đó, một Classification Head ngẫu nhiên sẽ được gắn lên trên DistilBERT base model.

Cấu hình `TrainingArguments`:
- `num_train_epochs`: 2
- `per_device_train_batch_size`: 8
- `per_device_eval_batch_size`: 8
- `learning_rate`: 2e-5
- `evaluation_strategy` / `eval_strategy`: epoch

In [7]:
# Cell 7 - Load Model & Thiết lập TrainingArguments
from transformers import AutoModelForSequenceClassification, TrainingArguments

# 1. Load Pretrained Model với 2 nhãn đầu ra
model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2
)

# 2. Cấu hình Tham số Huấn luyện (TrainingArguments)
eval_param_key = "eval_strategy" if hasattr(TrainingArguments("test"), "eval_strategy") else "evaluation_strategy"
kwargs = {
    "output_dir": "./results",
    "num_train_epochs": 2,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "save_strategy": "epoch",
    eval_param_key: "epoch",
    "report_to": "none"
}

training_args = TrainingArguments(**kwargs)
print("Đã khởi tạo model và TrainingArguments thành công!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đã khởi tạo model và TrainingArguments thành công!


### Bước 7 & 8: Định nghĩa Đánh giá Metrics & Khởi tạo Trainer
Hàm `compute_metrics` sẽ nhận các logits và labels, dùng `scikit-learn` để tính các chỉ số:
- **Accuracy**: Tỉ lệ dự đoán đúng toàn cục.
- **Precision**: Độ chính xác của các dự đoán tích cực.
- **Recall**: Khả năng bao phủ các mẫu tích cực thật sự.
- **F1-score**: Trung bình hài hòa giữa Precision và Recall.

In [8]:
# Cell 8 - Định nghĩa compute_metrics & Khởi tạo Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Khởi tạo Trainer thành công!")

Khởi tạo Trainer thành công!


### Bước 9 & 10: Huấn luyện (Fine-tuning) & Đánh giá (Evaluation)
Tiến hành gọi `trainer.train()` để thực hiện vòng lặp Forward pass -> Loss -> Backpropagation -> Update weights.
Sau đó gọi `trainer.evaluate()` để lấy báo cáo chỉ số đánh giá trên tập Test.

In [9]:
# Cell 9 - Chạy Fine-tuning và Đánh giá Model
print("=== BẮT ĐẦU QUÁ TRÌNH FINE-TUNING ===")
train_result = trainer.train()
print("\nFine-tuning hoàn tất!")

print("\n=== ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST ===")
eval_results = trainer.evaluate()

print(f"\nLoss trên tập eval (eval_loss):      {eval_results['eval_loss']:.4f}")
print(f"Độ chính xác (eval_accuracy):        {eval_results['eval_accuracy']:.4f} ({eval_results['eval_accuracy']*100:.2f}%)")
print(f"Chỉ số F1 (eval_f1):                  {eval_results['eval_f1']:.4f}")
print(f"Chỉ số Precision (eval_precision):   {eval_results['eval_precision']:.4f}")
print(f"Chỉ số Recall (eval_recall):         {eval_results['eval_recall']:.4f}")

=== BẮT ĐẦU QUÁ TRÌNH FINE-TUNING ===


C:\Users\Admin\Documents\GitHub\Practice 3\.venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.684155,0.675032,0.580000,0.086957,1.000000,0.045455
2,0.658785,0.655181,0.640000,0.357143,0.833333,0.227273


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Admin\Documents\GitHub\Practice 3\.venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning hoàn tất!

=== ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST ===


C:\Users\Admin\Documents\GitHub\Practice 3\.venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.658785,0.655181,2,0.640000,0.357143,0.833333,0.227273



Loss trên tập eval (eval_loss):      0.6552
Độ chính xác (eval_accuracy):        0.6400 (64.00%)
Chỉ số F1 (eval_f1):                  0.3571
Chỉ số Precision (eval_precision):   0.8333
Chỉ số Recall (eval_recall):         0.2273


### Bước 11: Thử nghiệm Model vừa Fine-tune với câu mới
Ta đưa 2 câu mới chưa từng thấy vào model vừa được fine-tune để xem kết quả dự đoán cảm xúc (POSITIVE / NEGATIVE):
1. *"This movie was absolutely amazing with brilliant acting!"*
2. *"I regret watching this terrible movie, it was a waste of time."*

In [10]:
# Cell 10 - Dự đoán câu mới với Fine-tuned Model
import torch

new_sentences = [
    "This movie was absolutely amazing with brilliant acting!",
    "I regret watching this terrible movie, it was a waste of time."
]

# Chuyển model sang chế độ đánh giá
model.eval()

print("=== KẾT QUẢ DỰ ĐOÁN VỚI FINE-TUNED MODEL ===")
for sentence in new_sentences:
    # 1. Tokenize câu đầu vào
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # 2. Đưa qua model lấy Logits
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        
    # 3. Tính xác suất Softmax & chọn label lớn nhất
    probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]
    predicted_class_id = torch.argmax(probabilities).item()
    label_map = {0: "NEGATIVE", 1: "POSITIVE"}
    
    predicted_label = label_map[predicted_class_id]
    confidence = probabilities[predicted_class_id].item()
    
    print(f"\nCâu: '{sentence}'")
    print(f"   => Dự đoán (Predicted Label): {predicted_label}")
    print(f"   => Độ tin cậy (Confidence):     {confidence:.4f} ({confidence*100:.2f}%)")

=== KẾT QUẢ DỰ ĐOÁN VỚI FINE-TUNED MODEL ===

Câu: 'This movie was absolutely amazing with brilliant acting!'
   => Dự đoán (Predicted Label): NEGATIVE
   => Độ tin cậy (Confidence):     0.5006 (50.06%)

Câu: 'I regret watching this terrible movie, it was a waste of time.'
   => Dự đoán (Predicted Label): NEGATIVE
   => Độ tin cậy (Confidence):     0.5631 (56.31%)


---
## 4. BẢNG SO SÁNH EXERCISE 1 VÀ EXERCISE 2 & KẾT LUẬN

### Bảng so sánh phương pháp:

| Tiêu chí | Exercise 1 (Pre-trained Pipeline) | Exercise 2 (Fine-tuning Base Model) |
|---|---|---|
| **Mô hình sử dụng** | `distilbert-base-uncased-finetuned-sst-2` (đã fine-tune sẵn) | `distilbert-base-uncased` (chưa có head phân loại) |
| **Quá trình huấn luyện** | Không huấn luyện (Zero-shot Inference) | Có huấn luyện thêm (Fine-tuning trên IMDb) |
| **Yêu cầu Dataset** | Không yêu cầu dataset lớn | Cần dataset cụ thể (Tập IMDb train/test) |
| **Tùy biến Task** | Phụ thuộc vào task của pre-trained model | Tùy biến linh hoạt theo bài toán của cá nhân |
| **Thời gian thực thi** | Rất nhanh (vài giây) | Tốn thời gian huấn luyện tùy thiết bị (vài phút/giờ) |
| **Độ phức tạp** | Rất đơn giản (chỉ với `pipeline`) | Trung bình (Tokenizer + Model + Trainer + Metrics) |

### Kết luận (Conclusion):
Thực hành Practice 3 đã giúp nắm vững cách áp dụng Hugging Face Transformers trong bài toán Xử lý Ngôn ngữ Tự nhiên (NLP). Việc sử dụng mô hình pre-trained thông qua `pipeline` (Exercise 1) giải quyết cực kỳ nhanh chóng các bài toán chuẩn mà không cần huấn luyện lại. Trong khi đó, kỹ thuật **Fine-tuning** (Exercise 2) cho phép kế thừa tri thức tổng quát của mô hình ngôn ngữ lớn (LLM/Pre-trained base model) và tinh chỉnh Classification Head để đạt hiệu năng cao trên dữ liệu chuyên biệt.